# Path 0: Product catalog lifecycle

Covers `POST/GET/GET :id/PUT/PATCH/DELETE /products`.

`products` is the catalog entity (`sku`, `name`, `unit_price_cents`). Order line
items reference a product by `product_id`; deleting a product that is still
referenced by an order-item returns 409.

Uses Faker to create several catalog rows, then exercises CRUD and the FK guard.

Run top-to-bottom (e.g. `jupyter nbconvert --to notebook --execute 00_product_catalog_flow.ipynb`).

In [ ]:
import os
import random

import requests
from faker import Faker

BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:3002/api")
fake = Faker()
CATALOG_COUNT = int(os.environ.get("CATALOG_COUNT", "6"))


def fake_product():
    return {
        "sku": f"SKU-{fake.unique.bothify(text='???-####').upper()}",
        "name": fake.unique.catch_phrase(),
        "unit_price_cents": fake.random_int(min=99, max=14999),
    }


print(f"BASE_URL={BASE_URL} CATALOG_COUNT={CATALOG_COUNT}")

## Step 1 - Bulk-create catalog products with Faker

In [ ]:
catalog = []
for _ in range(CATALOG_COUNT):
    resp = requests.post(f"{BASE_URL}/products", json=fake_product())
    assert resp.status_code == 201, resp.text
    catalog.append(resp.json())

print(f"Created {len(catalog)} products")
catalog[:3]

## Step 2 - List products and fetch one by id

In [ ]:
resp = requests.get(f"{BASE_URL}/products")
assert resp.status_code == 200, resp.text
all_products = resp.json()
catalog_ids = {p["id"] for p in catalog}
assert catalog_ids.issubset({p["id"] for p in all_products})

target = random.choice(catalog)
resp = requests.get(f"{BASE_URL}/products/{target['id']}")
assert resp.status_code == 200, resp.text
fetched = resp.json()
assert fetched["sku"] == target["sku"]
print("Fetched product:", fetched)

## Step 3 - Full update (PUT) then partial update (PATCH)

In [ ]:
put_payload = fake_product()
resp = requests.put(f"{BASE_URL}/products/{target['id']}", json=put_payload)
assert resp.status_code == 200, resp.text
updated = resp.json()
assert updated["sku"] == put_payload["sku"]
assert updated["name"] == put_payload["name"]
target = updated
print("Updated (PUT) product:", updated)

patched_name = fake.unique.catch_phrase()
resp = requests.patch(f"{BASE_URL}/products/{target['id']}", json={"name": patched_name})
assert resp.status_code == 200, resp.text
patched = resp.json()
assert patched["name"] == patched_name
assert patched["sku"] == target["sku"]
target = patched
print("Patched product:", patched)

## Step 4 - FK guard: DELETE while referenced by an order-item returns 409

In [ ]:
resp = requests.post(
    f"{BASE_URL}/customers",
    json={"email": fake.unique.email(), "password": fake.password(length=14)},
)
assert resp.status_code == 201, resp.text
customer = resp.json()

resp = requests.post(f"{BASE_URL}/orders", json={"customer_id": customer["id"]})
assert resp.status_code == 201, resp.text
order = resp.json()

resp = requests.post(
    f"{BASE_URL}/order-items",
    json={
        "order_id": order["id"],
        "product_id": target["id"],
        "quantity": 1,
        "unit_price_cents": target["unit_price_cents"],
    },
)
assert resp.status_code == 201, resp.text
item = resp.json()

resp = requests.delete(f"{BASE_URL}/products/{target['id']}")
assert resp.status_code == 409, f"expected 409 while referenced, got {resp.status_code} {resp.text}"
print("Confirmed 409 while product is referenced by an order-item")

## Step 5 - After the line item is gone, DELETE the product succeeds

In [ ]:
resp = requests.delete(f"{BASE_URL}/customers/{customer['id']}")
assert resp.status_code == 200, resp.text

resp = requests.delete(f"{BASE_URL}/products/{target['id']}")
assert resp.status_code == 200, resp.text
print("Deleted product:", resp.json())

resp = requests.get(f"{BASE_URL}/products/{target['id']}")
assert resp.status_code == 404, f"expected 404 after delete, got {resp.status_code}"
print("Confirmed 404 after product delete")